In [1]:
import os, re, glob, itertools, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

COMP = "/kaggle/input/competitions/ai-of-god-5/participant_data"
df = pd.read_csv(f"{COMP}/train.csv"); te = pd.read_csv(f"{COMP}/sample_submission.csv")
le = LabelEncoder(); y = le.fit_transform(df.animal_class); NC = len(le.classes_)
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(y)), y))
WORK = "/kaggle/working"
print(len(y), NC)

3087 72


In [2]:
files = {}
for root, _, fs in os.walk("/kaggle/input"):
    for f in fs:
        if f.endswith(".npy"): files[f] = os.path.join(root, f)

runs = {}
for f, p in files.items():
    m = re.match(r"oof_(.+)_f(\d)\.npy$", f)
    if m: runs.setdefault(m.group(1), {})[int(m.group(2))] = p
print("fine-tuned runs:", {t: sorted(v) for t, v in runs.items()})
RES = sorted(int(re.match(r"Ftr(\d+)\.npy$", f).group(1)) for f in files if re.match(r"Ftr(\d+)\.npy$", f))
print("frozen resolutions:", RES, "| with test features:", [r for r in RES if f"Fte{r}.npy" in files])

fine-tuned runs: {'D3L560': [0, 1], 'D3L336': [0, 1, 2, 3, 4], 'D3L448': [0, 1], 'EVA448': [0, 1]}
frozen resolutions: [336, 448, 672] | with test features: [336, 448, 672]


In [3]:
def sib(p, a, b): return os.path.join(os.path.dirname(p), os.path.basename(p).replace(a, b, 1))
OOF, TEST, ROWS = {}, {}, {}
EXCLUDE = set()                     # e.g. {"EVA448", "D3S336ml"} to leave known-weak runs out of the search

for tag, fd in runs.items():
    if tag in EXCLUDE: continue
    o = np.full((len(y), NC), np.nan, np.float32); tests = []
    for k, p in sorted(fd.items()):
        vi, ts = sib(p, "oof_", "va_idx_"), sib(p, "oof_", "test_")
        if not (os.path.exists(vi) and os.path.exists(ts)): print("incomplete:", tag, "fold", k); continue
        o[np.load(vi)] = np.load(p); tests.append(np.load(ts))
    OOF[tag], TEST[tag], ROWS[tag] = o, np.mean(tests, 0), ~np.isnan(o[:, 0])
    print(f"{tag:<12} folds {len(tests)} | rows {ROWS[tag].sum():>5} | errors on own rows {(o[ROWS[tag]].argmax(1) != y[ROWS[tag]]).sum()}")

D3L560       folds 2 | rows  1236 | errors on own rows 16
D3L336       folds 5 | rows  3087 | errors on own rows 52
D3L448       folds 2 | rows  1236 | errors on own rows 14
EVA448       folds 2 | rows  1236 | errors on own rows 44


In [4]:
def lr_block(res):
    Ftr, Fte = np.load(files[f"Ftr{res}.npy"]), np.load(files[f"Fte{res}.npy"])
    oof = np.full((len(y), NC), np.nan, np.float32)
    for tr, va in folds:
        sc = StandardScaler().fit(Ftr[tr])
        oof[va] = LogisticRegression(C=0.5, max_iter=2000).fit(sc.transform(Ftr[tr]), y[tr]).predict_proba(sc.transform(Ftr[va]))
    sc = StandardScaler().fit(Ftr)
    tp = LogisticRegression(C=0.5, max_iter=2000).fit(sc.transform(Ftr), y).predict_proba(sc.transform(Fte))
    return oof, tp

if os.path.exists(f"{WORK}/frozen_oof.npy"):
    OOF["FROZEN"], TEST["FROZEN"] = np.load(f"{WORK}/frozen_oof.npy"), np.load(f"{WORK}/frozen_test.npy")
else:
    parts = [lr_block(r) for r in (336, 448, 672)]
    OOF["FROZEN"], TEST["FROZEN"] = np.mean([p[0] for p in parts], 0), np.mean([p[1] for p in parts], 0)
    np.save(f"{WORK}/frozen_oof.npy", OOF["FROZEN"]); np.save(f"{WORK}/frozen_test.npy", TEST["FROZEN"])
ROWS["FROZEN"] = np.ones(len(y), bool)
print("FROZEN errors:", int((OOF["FROZEN"].argmax(1) != y).sum()))

FROZEN errors: 56


In [5]:
BASE = ["D3L336", "FROZEN"]
assert all(b in OOF for b in BASE), f"missing component, available: {list(OOF)}"
comps = list(OOF)

def errs(names, rows):
    P = np.mean([OOF[n] for n in names], 0); return P[rows].argmax(1) != y[rows]

results = []
for r in range(1, len(comps) + 1):
    for sub in itertools.combinations(comps, r):
        rows = np.all([ROWS[n] for n in set(sub) | set(BASE)], axis=0)
        eb, ec = errs(BASE, rows), errs(sub, rows)
        results.append(dict(ensemble="+".join(sub), rows=int(rows.sum()), base_err=int(eb.sum()), err=int(ec.sum()),
                            fixed=int((eb & ~ec).sum()), broken=int((~eb & ec).sum())))
R = pd.DataFrame(results); R["gain"] = R.base_err - R.err
print(R.sort_values(["gain", "rows"], ascending=[False, False]).head(15).to_string(index=False))

                          ensemble  rows  base_err  err  fixed  broken  gain
                     D3L336+D3L448  1236        19   12      7       0     7
              D3L560+D3L336+D3L448  1236        19   12      8       1     7
              D3L560+EVA448+FROZEN  1236        19   12      9       2     7
       D3L560+D3L448+EVA448+FROZEN  1236        19   12      8       1     7
D3L560+D3L336+D3L448+EVA448+FROZEN  1236        19   12      8       1     7
       D3L560+D3L336+EVA448+FROZEN  1236        19   13      7       1     6
                            D3L448  1236        19   14      8       3     5
                     D3L560+D3L448  1236        19   14      7       2     5
              D3L560+D3L448+FROZEN  1236        19   14      6       1     5
              D3L448+EVA448+FROZEN  1236        19   14      6       1     5
       D3L560+D3L336+D3L448+EVA448  1236        19   14      8       3     5
       D3L560+D3L336+D3L448+FROZEN  1236        19   14      5       0     5

In [6]:
CHOSEN = ["D3L448", "EVA448", "D3L560", "D3L336", "FROZEN"]
P = np.mean([TEST[n] for n in CHOSEN], 0)
sub = pd.DataFrame({"image_id": te.image_id, "animal_class": le.classes_[P.argmax(1)]})
sub.to_csv(f"{WORK}/submission_5ensemble.csv", index=False)

cur = np.mean([TEST[n] for n in ["D3L336", "FROZEN"]], 0).argmax(1)     # your last submitted blend
alone = TEST["D3L336"].argmax(1)
print(sub.shape, sub.animal_class.nunique(), "classes")
print("labels changed vs your current blend:", int((P.argmax(1) != cur).sum()), "| vs D3L336 alone:", int((P.argmax(1) != alone).sum()))

(1179, 2) 72 classes
labels changed vs your current blend: 5 | vs D3L336 alone: 3


In [7]:
!pip -q install -U timm
import os, gc, numpy as np, pandas as pd, cv2, torch, timm
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

DEV = "cuda"; COMP = "/kaggle/input/competitions/ai-of-god-5/participant_data"; OUT = "/kaggle/working"
ARCH = "vit_large_patch16_dinov3.lvd1689m"
df = pd.read_csv(f"{COMP}/train.csv"); te = pd.read_csv(f"{COMP}/sample_submission.csv")
le = LabelEncoder(); y = le.fit_transform(df.animal_class); NC = len(le.classes_)
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(y)), y))
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)

def resolve(d, n):
    p = f"{COMP}/{d}/{n}"
    if os.path.exists(p): return p
    for e in (".png", ".jpg", ".jpeg"):
        if os.path.exists(p + e): return p + e
    raise FileNotFoundError(p)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 68.4 MB/s eta 0:00:00


In [8]:
def letterbox(p, S):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]; s = S / max(h, w)
    im = cv2.resize(im, (max(1, round(w*s)), max(1, round(h*s))), interpolation=cv2.INTER_CUBIC)
    c = np.zeros((S, S, 3), np.uint8); oy, ox = (S - im.shape[0])//2, (S - im.shape[1])//2
    c[oy:oy+im.shape[0], ox:ox+im.shape[1]] = im; return c

@torch.no_grad()
def extract(paths, FS, bs):
    m = timm.create_model(ARCH, pretrained=True, num_classes=0, img_size=FS).to(DEV).eval(); out = []
    for i in tqdm(range(0, len(paths), bs)):
        x = torch.from_numpy(np.stack([letterbox(p, FS) for p in paths[i:i+bs]])).to(DEV).permute(0, 3, 1, 2).float() / 255
        x = (x - MEAN) / STD; acc = 0
        for fl in (0, 1):
            with torch.autocast("cuda", dtype=torch.float16): ft = m.forward_features(x.flip(3) if fl else x).float()
            acc = acc + torch.cat([ft[:, 0], ft[:, m.num_prefix_tokens:].mean(1)], 1)
        out.append((acc / 2).cpu())
    del m; gc.collect(); torch.cuda.empty_cache()
    return torch.cat(out).numpy()

for FS, bs in [(560, 4), (784, 2)]:                   # 784 is heavy: stop it if the first batches look too slow
    if os.path.exists(f"{OUT}/Fte{FS}.npy"): print(FS, "exists"); continue
    try:
        np.save(f"{OUT}/Ftr{FS}.npy", extract([resolve("train_images", n) for n in df.image_id], FS, bs))
        np.save(f"{OUT}/Fte{FS}.npy", extract([resolve("test_images", n) for n in te.image_id], FS, bs))
        print("saved", FS)
    except torch.cuda.OutOfMemoryError:
        print("out of memory at", FS, "- skipping"); gc.collect(); torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

  0%|          | 0/772 [00:00<?, ?it/s]

  0%|          | 0/295 [00:00<?, ?it/s]

saved 560


  0%|          | 0/1544 [00:00<?, ?it/s]

  0%|          | 0/590 [00:00<?, ?it/s]

saved 784


In [9]:
files = {}
for base in ("/kaggle/input", OUT):
    for root, _, fs in os.walk(base):
        for f in fs:
            if f.endswith(".npy"): files[f] = os.path.join(root, f)
RES = sorted(int(f[3:-4]) for f in files if f.startswith("Ftr") and f"Fte{f[3:-4]}.npy" in files)
print("resolutions:", RES)

def knn_proba(Ftr, ytr, Fq, k=10, T=0.05):
    a = Ftr / np.linalg.norm(Ftr, axis=1, keepdims=True); b = Fq / np.linalg.norm(Fq, axis=1, keepdims=True)
    S = b @ a.T; idx = np.argpartition(-S, k, axis=1)[:, :k]; P = np.zeros((len(Fq), NC), np.float32)
    for i in range(len(Fq)):
        s = S[i, idx[i]]; np.add.at(P[i], ytr[idx[i]], np.exp((s - s.max()) / T))
    return P / P.sum(1, keepdims=True)

def heads(Ftr, Fq, ytr):
    sc = StandardScaler().fit(Ftr); a, b = sc.transform(Ftr), sc.transform(Fq)
    return (LogisticRegression(C=0.5, max_iter=2000).fit(a, ytr).predict_proba(b), knn_proba(a, ytr, b))

OOF = {"lr": {}, "knn": {}}; TST = {"lr": {}, "knn": {}}
for r in RES:
    Ftr, Fte = np.load(files[f"Ftr{r}.npy"]), np.load(files[f"Fte{r}.npy"])
    ol, ok = np.zeros((len(y), NC), np.float32), np.zeros((len(y), NC), np.float32)
    for tr, va in folds:
        ol[va], ok[va] = heads(Ftr[tr], Ftr[va], y[tr])
    TST["lr"][r], TST["knn"][r] = heads(Ftr, Fte, y)
    OOF["lr"][r], OOF["knn"][r] = ol, ok
    print(f"{r}: LR errors {(ol.argmax(1) != y).sum()} | kNN errors {(ok.argmax(1) != y).sum()}")

for h in ("lr", "knn"):
    o = np.mean(list(OOF[h].values()), 0); print(f"all-res {h}: errors {(o.argmax(1) != y).sum()}")
    np.save(f"{OUT}/frozen_{h}_oof.npy", o); np.save(f"{OUT}/frozen_{h}_test.npy", np.mean(list(TST[h].values()), 0))

resolutions: [336, 448, 560, 672, 784]
336: LR errors 71 | kNN errors 98
448: LR errors 63 | kNN errors 99
560: LR errors 59 | kNN errors 94
672: LR errors 56 | kNN errors 90
784: LR errors 52 | kNN errors 94
all-res lr: errors 54
all-res knn: errors 90
